# V2-01b — Cascade **validation** (Methods A–D): is the cascade REAL or an ARTIFACT?

**The make-or-break notebook.** It renders the V2-S04 validation of the V2-S03 cascade:
null/permutation (Method B), drop-one-journal jackknife of origins (Method C), topic-granularity
sensitivity (Method D), and held-out years (Method A, supporting). It enters **no hypothesis
gate** — it is the primary input to the human spend gate **G6**.

> ## ⚠️ 1995 left-censoring (read before interpreting anything below)
> The corpus starts in 1995, so **~80% of topics are "all-tied at 1995"** — present in several
> journals at panel start with no resolvable ordering. A 1995 origin means **"present at panel
> start"**, never "born in 1995". Two defenses: (1) the primary null (**Null-1**, journal-label
> shuffle *within topic*) preserves each topic's year footprint, so the all-tied topics
> contribute the *same* zero ordering to both observed and null and cannot inflate significance;
> (2) every verdict is recomputed on the **resolvable subset** (topics with ≥ 2 distinct
> first-appearance years). If the structure lived only in the 1995 ties it would vanish there.
> **It does not.**

**Headline verdict: REAL** on the gate-critical checks (S1 seeding-spread null + origin jackknife
+ granularity), with a documented **PARTIAL** qualification on the secondary lead-lag-asymmetry
statistic (S2) and the short held-out window. See `V2/docs/cartography/cascade_validation.md`.

## 1. Setup + load the locked (1000-perm) verdict tables

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

# Repo-root sniff — this notebook lives at V2/notebooks/, code is under src/.
repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from scifield.cartography import cascade_validation as cv  # noqa: E402

CASCADE_DIR = repo_root / "V2" / "data" / "cascade"
FLOW_DIR = repo_root / "V2" / "data" / "flow"

null_tbl = pd.read_parquet(CASCADE_DIR / "null_results.parquet")
jk_tbl = pd.read_parquet(CASCADE_DIR / "jackknife_results.parquet")
gran_tbl = pd.read_parquet(CASCADE_DIR / "granularity_results.parquet")

flow_leaf = pd.read_parquet(FLOW_DIR / "flow_leaf.parquet")
flow_mid = pd.read_parquet(FLOW_DIR / "flow_mid.parquet")

print("LOCKED constants: n_perm =", cv.N_PERM, " seed =", cv.SEED)
print(
    "null_results:",
    null_tbl.shape,
    " jackknife_results:",
    jk_tbl.shape,
    " granularity_results:",
    gran_tbl.shape,
)
null_tbl[
    [
        "grain",
        "subset",
        "stat",
        "null",
        "observed",
        "z",
        "p_value",
        "cell_verdict",
        "method_b_verdict",
    ]
]

LOCKED constants: n_perm = 1000  seed = 20260609
null_results: (16, 14)  jackknife_results: (4, 9)  granularity_results: (3, 8)


,grain,subset,stat,null,observed,z,p_value,cell_verdict,method_b_verdict
0,leaf,all,S1,within_topic,0.090674,3.273446,0.001998,PASS,PARTIAL
1,leaf,all,S1,year_shuffle,0.090674,4.721215,0.000999,PASS,PARTIAL
2,leaf,all,S2,within_topic,6.798069,0.407878,0.356643,FAIL,PARTIAL
3,leaf,all,S2,year_shuffle,6.798069,3.347806,0.001998,PASS,PARTIAL
4,leaf,resolvable,S1,within_topic,0.090679,3.215284,0.001998,PASS,PARTIAL
5,leaf,resolvable,S1,year_shuffle,0.090679,5.200106,0.000999,PASS,PARTIAL
6,leaf,resolvable,S2,within_topic,6.971969,0.602588,0.276723,FAIL,PARTIAL
7,leaf,resolvable,S2,year_shuffle,6.971969,3.312532,0.003996,PASS,PARTIAL
8,mid,all,S1,within_topic,0.109661,4.040629,0.000999,PASS,PARTIAL
9,mid,all,S1,year_shuffle,0.109661,5.030676,0.000999,PASS,PARTIAL


## 2. Method B — null distribution vs observed (the gate-critical panel)

The primary, gate-critical cell is **S1 (seeding-score spread) under Null-1 (journal-label
shuffle within topic)** at the leaf grain. We recompute a fresh null distribution here (smaller
`n_perm` for notebook speed) and draw the observed value as a line — the locked p/z come from the
1000-perm table loaded above. The same panel is drawn for the **resolvable subset** to show the
structure is not a 1995-tie artifact.

In [2]:
N_PERM_NB = 400  # notebook-speed null; the LOCKED verdict uses cv.N_PERM = 1000 (table above)


def null_distribution(flow, *, topic_key, stat, null, n_perm, seed, resolvable_only):
    """Recompute the observed stat + its null replicates for plotting."""
    from scifield.cartography import nulls
    from scifield.cartography.cascade import _first_appearance_frame

    work = flow
    if resolvable_only:
        keep = set(cv.resolvable_topics(work, topic_key=topic_key))
        work = work[work[topic_key].isin(keep)]
    stat_fn = cv._STATS[stat]
    observed = stat_fn(work, topic_key=topic_key)
    if null == "within_topic":
        base, label_col = work, "journal_slug"
    else:
        base, label_col = _first_appearance_frame(work, topic_key=topic_key), "first_year"
    reps = np.array(
        [
            stat_fn(
                nulls.permute_labels(base, label_col=label_col, group_col=topic_key, seed=seed + i),
                topic_key=topic_key,
            )
            for i in range(n_perm)
        ]
    )
    return observed, reps


fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, resolvable in zip(axes, [False, True], strict=False):
    obs, reps = null_distribution(
        flow_leaf,
        topic_key="topic_id",
        stat="S1",
        null="within_topic",
        n_perm=N_PERM_NB,
        seed=cv.SEED,
        resolvable_only=resolvable,
    )
    want_subset = "resolvable" if resolvable else "all"
    row = null_tbl[
        (null_tbl["grain"] == "leaf")
        & (null_tbl["subset"] == want_subset)
        & (null_tbl["stat"] == "S1")
        & (null_tbl["null"] == "within_topic")
    ].iloc[0]
    ax.hist(reps, bins=30, color="#b0c4de", edgecolor="white", label="Null-1 replicates")
    ax.axvline(obs, color="#c0392b", lw=2.5, label=f"observed = {obs:.4f}")
    ax.axvline(
        reps.mean(), color="#7f8c8d", lw=1.2, ls="--", label=f"null mean = {reps.mean():.4f}"
    )
    subset = "resolvable subset (>=2 first-years)" if resolvable else "all topics"
    ax.set_title(
        f"S1 seeding-spread / Null-1 — leaf, {subset}\n"
        f"LOCKED: z={row.z:+.2f}, p={row.p_value:.4f} ({row.cell_verdict})"
    )
    ax.set_xlabel("seeding-score spread (std across 10 journals)")
    ax.set_ylabel("null replicate count")
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(repo_root / "V2" / "notebooks" / "figures" / "v2_01b_null_s1.png", dpi=110)
plt.show()
print(
    "Observed S1 sits in the extreme upper tail of the within-topic shuffle null at BOTH "
    "all-topics and resolvable — the cascade is not a 1995-censoring artifact."
)

Observed S1 sits in the extreme upper tail of the within-topic shuffle null at BOTH all-topics and resolvable — the cascade is not a 1995-censoring artifact.


/var/folders/px/cf1lhf6n1w3cdhxtl6k9xglw0000gn/T/ipykernel_67409/4188612038.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Why S2 (secondary) does not corroborate under Null-1

S2 (net lead-lag *asymmetry magnitude*) is largely **conserved** by the within-topic label
shuffle: Null-1 reassigns which journal holds each year but preserves the *set* of years per
topic, so the total pairwise asymmetry barely moves. It destroys *who leads whom* (which S1
catches) but not *how much total asymmetry* exists (which S2 sums). Under Null-2 (year shuffle)
S2 passes. S1 is the protocol's designated primary; this is a known instrument mismatch, not a
failure of the cascade.

In [3]:
fig, ax = plt.subplots(figsize=(9, 4))
cells = null_tbl.query("grain=='leaf' and subset=='all'").copy()
cells["label"] = cells["stat"] + " / " + cells["null"]
colors = [
    "#27ae60" if v == "PASS" else "#f39c12" if v == "PARTIAL" else "#c0392b"
    for v in cells["cell_verdict"]
]
ax.barh(cells["label"], cells["z"], color=colors)
ax.axvline(2.0, color="#2c3e50", ls="--", lw=1, label="z = 2 (PASS bar)")
ax.set_xlabel("z-score (observed vs null)")
ax.set_title("Method B cells (leaf, all topics): green=PASS, orange=PARTIAL, red=FAIL")
for y, (z, p) in enumerate(zip(cells["z"], cells["p_value"], strict=False)):
    ax.text(z + 0.05, y, f"p={p:.3f}", va="center", fontsize=8)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(repo_root / "V2" / "notebooks" / "figures" / "v2_01b_methodb_cells.png", dpi=110)
plt.show()

/var/folders/px/cf1lhf6n1w3cdhxtl6k9xglw0000gn/T/ipykernel_67409/3409790654.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Method C — drop-one-journal jackknife of origin attributions

`attribution_flip_rate` is the fraction of (topic, held-out) cells whose origin *changes* in a
**non-forced** run (a topic whose origin journal IS the dropped journal is *forced* to
reattribute and is excluded, counted separately as `forced_reattribution_count`). Flip rate is
**0.0 at every grain and subset** — origins are stable to panel composition.

In [4]:
per_run = cv.jackknife_origins(flow_leaf, topic_key="topic_id")
# Per-dropped-journal residual flip rate (non-forced changes vs full origin).
rows = []
for held, g in per_run.groupby("held_out"):
    elig = g[~g["forced"]]
    flips = int(
        (
            elig["origin_journal_slug"].astype(object).to_numpy()
            != elig["full_origin"].astype(object).to_numpy()
        ).sum()
    )
    forced = int(g["forced"].sum())
    rows.append(
        {"held_out": held, "residual_flip_rate": flips / max(len(elig), 1), "forced": forced}
    )
per_journal = pd.DataFrame(rows).sort_values("held_out")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    per_journal["held_out"],
    per_journal["residual_flip_rate"],
    color="#5dade2",
    label="residual flip rate (non-forced)",
)
ax.bar(per_journal["held_out"], [0] * len(per_journal), bottom=0)
ax.axhline(0.20, color="#c0392b", ls="--", lw=1, label="PASS bar (0.20)")
for x, f in zip(range(len(per_journal)), per_journal["forced"], strict=False):
    ax.text(x, 0.005, f"forced={f}", ha="center", va="bottom", fontsize=7, rotation=90)
ax.set_ylim(0, 0.25)
ax.set_ylabel("origin attribution_flip_rate")
ax.set_title("Method C — drop-one-journal jackknife of leaf origins (flip rate = 0.0 everywhere)")
ax.set_xticklabels(per_journal["held_out"], rotation=45, ha="right")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(repo_root / "V2" / "notebooks" / "figures" / "v2_01b_jackknife.png", dpi=110)
plt.show()
jk_tbl[
    [
        "grain",
        "subset",
        "attribution_flip_rate",
        "forced_reattribution_count",
        "n_genuine_flips",
        "verdict",
    ]
]

/var/folders/px/cf1lhf6n1w3cdhxtl6k9xglw0000gn/T/ipykernel_67409/1869859145.py:23: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(per_journal["held_out"], rotation=45, ha="right")
/var/folders/px/cf1lhf6n1w3cdhxtl6k9xglw0000gn/T/ipykernel_67409/1869859145.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,grain,subset,attribution_flip_rate,forced_reattribution_count,n_genuine_flips,verdict
0,leaf,all,0.0,149,0,PASS
1,leaf,resolvable,0.0,138,0,PASS
2,mid,all,0.0,96,0,PASS
3,mid,resolvable,0.0,86,0,PASS


## 4. Method D — topic-granularity sensitivity (leaf vs mid)

Per-journal seeding score at the leaf grain (149 topics) vs the mid grain (96 topics). PASS if
Spearman ρ ≥ 0.5. **ρ = 0.85** — the lead/follow ranking is robust to topic resolution.

In [5]:
res_d = cv.granularity_consistency(flow_leaf, flow_mid)
leaf_s = pd.Series(res_d["leaf_scores"])
mid_s = pd.Series(res_d["mid_scores"])
paired = pd.concat([leaf_s.rename("leaf"), mid_s.rename("mid")], axis=1).dropna()

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(paired["leaf"], paired["mid"], s=60, color="#8e44ad")
for j, r in paired.iterrows():
    ax.annotate(j, (r["leaf"], r["mid"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
lims = [min(paired.min()) - 0.02, max(paired.max()) + 0.02]
ax.plot(lims, lims, color="#bdc3c7", ls="--", lw=1)
ax.set_xlabel("leaf-grain seeding score")
ax.set_ylabel("mid-grain seeding score")
ax.set_title(
    f"Method D — granularity (leaf vs mid)\nSpearman rho = {res_d['spearman_rho']:.3f} "
    f"({res_d['verdict']})"
)
fig.tight_layout()
fig.savefig(repo_root / "V2" / "notebooks" / "figures" / "v2_01b_granularity.png", dpi=110)
plt.show()

/var/folders/px/cf1lhf6n1w3cdhxtl6k9xglw0000gn/T/ipykernel_67409/790687418.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Method A — held-out years (1995–2018 → 2019–2025) [SUPPORTING ONLY]

Per-journal seeding ranking learned on 1995–2018 vs recomputed on 2019–2025. **Supporting only**
— a PARTIAL/FAIL here does not sink the gate (7-year window, left-censored). Leaf ρ = 0.30
(PARTIAL); mid ρ = 0.09 (FAIL). The ranking reproduces out of sample only partially — a
qualification, attributed to the short window + censoring, to revisit with the expanded corpus.

In [6]:
res_a = cv.heldout_consistency(flow_leaf, topic_key="topic_id")
tr = pd.Series(res_a["train_scores"])
ho = pd.Series(res_a["holdout_scores"])
paired_a = pd.concat([tr.rename("train"), ho.rename("holdout")], axis=1).dropna()

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(paired_a["train"], paired_a["holdout"], s=60, color="#16a085")
for j, r in paired_a.iterrows():
    ax.annotate(
        j, (r["train"], r["holdout"]), fontsize=7, xytext=(3, 3), textcoords="offset points"
    )
lims = [min(paired_a.min()) - 0.02, max(paired_a.max()) + 0.02]
ax.plot(lims, lims, color="#bdc3c7", ls="--", lw=1)
ax.set_xlabel("train (1995-2018) seeding score")
ax.set_ylabel("holdout (2019-2025) seeding score")
ax.set_title(
    f"Method A — held-out years (leaf, SUPPORTING)\nSpearman rho = "
    f"{res_a['spearman_rho']:.3f} ({res_a['verdict']}); "
    f"excluded {res_a['n_topics_excluded']} post-2018 topics"
)
fig.tight_layout()
fig.savefig(repo_root / "V2" / "notebooks" / "figures" / "v2_01b_heldout.png", dpi=110)
plt.show()
gran_tbl

/var/folders/px/cf1lhf6n1w3cdhxtl6k9xglw0000gn/T/ipykernel_67409/736523107.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,method,grain,spearman_rho,bar,n_compared,n_excluded,supporting_only,verdict
0,D_granularity,leaf_vs_mid,0.854545,0.5,10,0,False,PASS
1,A_heldout,leaf,0.296970,0.5,10,1,True,PARTIAL
2,A_heldout,mid,0.090909,0.5,10,0,True,FAIL


## 6. Verdict — REAL (with PARTIAL qualification)

| Method | gate-critical? | result | verdict |
|---|---|---|---|
| **B — S1 / Null-1** (seeding spread) | **YES (primary)** | z 3.2–4.0, p ≤ 0.002, **survives resolvable subset** | **PASS** |
| B — S1 / Null-2 | corroborating | z 4.7–5.2 | PASS |
| B — S2 / Null-1 | corroborating | z 0.4 leaf (conserved by shuffle) | FAIL → block PARTIAL |
| **C — origin jackknife** | **YES** | flip rate 0.0 (both grains, both subsets) | **PASS** |
| **D — granularity** | YES | leaf-vs-mid ρ = 0.85 | **PASS** |
| A — held-out years | supporting | leaf ρ 0.30 / mid ρ 0.09 | PARTIAL / FAIL |

**The inter-journal cascade is REAL, not a panel/temporal/1995-censoring artifact.** The
gate-critical primary (S1/Null-1) passes cleanly at both grains *and on the resolvable subset*,
origins are jackknife-stable, and the ranking is grain-robust. The qualifications: the secondary
S2 statistic does not corroborate under the primary null (conserved by the within-topic shuffle),
and the held-out check is only PARTIAL (short window + censoring).

**Scope split for G6:** S04 owns the **origin** jackknife + the permutation nulls (this notebook);
**V2-S05 owns the role-score jackknife (`role_rank_correlation`)** — the other half of Method C.
The G6 synthesis must combine both halves plus a robust V2-S06 novelty finding.